# RSNA Knee Abnormality Detection - V35: 320x320 Clean High-Res Baseline (Option A)

**Strategic Resolution Scaling:**
- Retains our proven champion backbone: **`EfficientNet-B2`**.
- Scales spatial resolution from $256\times 256$ to **$320\times 320$** (+56% pixel density) to resolve subtle hairline fractures, early cartilage fibrillation, and meniscal horn fissures.
- **Eliminates the V32 flaw**: Uses the exact proven **Weighted BCE + Baker Decoupled + Calibrated ReportTeacher (20%)** loss (completely avoiding the zero-gradient defect of Asymmetric Loss).

**Architecture & Training Specs:**
1. **Backbone**: `efficientnet_b2` with Multi-View Dynamic Attention Pooling.
2. **Resolution**: $320 \times 320$ (Adaptive 3-View Slicing: 8 Sag + 6 Cor + 6 Axi = 20 slices with CLAHE).
3. **Loss**: Masked BCE with Baker's cyst decoupled from backprop + Calibrated NLP ReportTeacher Distillation (20%).
4. **Optimization**: AdamW ($lr=2\times 10^{-4}$, weight decay = $2\times 10^{-2}$), Cosine Annealing, AMP mixed precision.
5. **Outputs**: Saves 5 trained checkpoints (`model_b2_320_seed42_fold0..4.pth`) for the upcoming **Multi-Resolution Mega-Ensemble (256x256 + 320x320)**.

In [ ]:
import os, re, gc, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

try:
    import cv2
    HAS_CV2 = True
    print('CLAHE: using OpenCV')
except ImportError:
    HAS_CV2 = False
    from PIL import ImageOps
    print('CLAHE: using PIL fallback')

## 1. Configuration & Offline Weights Setup

In [ ]:
class CFG:
    seed           = 42
    n_folds        = 5
    epochs         = 5
    batch_size     = 4
    accum_steps    = 4
    lr             = 2e-4
    lr_min         = 1e-6
    weight_decay   = 2e-2
    backbone       = 'efficientnet_b2'
    sag_slices     = 8
    cor_slices     = 6
    axi_slices     = 6
    total_slices   = 20
    image_size     = (320, 320)    # Option A: 320x320 high resolution (+56% pixel density)
    num_classes    = 12
    target_cols    = [
        'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
        'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
        'Synovitis', "Baker's", 'Contusion', 'Fracture'
    ]
    bakers_idx     = 9
    distill_weight = 0.20
    device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp        = True
    num_workers    = 4
    use_clahe      = True

# Auto-locate offline B2 weights
WEIGHTS_PATH = None
for wp in [
    Path(f'/kaggle/input/notebook00e3adcdf0/weights/{CFG.backbone}.pth'),
    Path(f'/kaggle/input/notebooks/arnabnandy2/notebook00e3adcdf0/weights/{CFG.backbone}.pth'),
    Path(f'/kaggle/input/notebooka2dd443e8c/weights/{CFG.backbone}.pth'),
    Path(f'/kaggle/input/notebooks/arnabnandy2/notebooka2dd443e8c/weights/{CFG.backbone}.pth'),
]:
    if wp.exists():
        WEIGHTS_PATH = wp; break

if WEIGHTS_PATH is None and Path('/kaggle/input').exists():
    for f in Path('/kaggle/input').rglob(f'{CFG.backbone}.pth'):
        WEIGHTS_PATH = f; break

print(f'Backbone       : {CFG.backbone}')
print(f'Resolution     : {CFG.image_size}')
print(f'Weights Path   : {WEIGHTS_PATH or "NOT FOUND (online fallback)"}')
print(f'Random Seed    : {CFG.seed}')
print(f'Slices Config  : {CFG.sag_slices} Sag + {CFG.cor_slices} Cor + {CFG.axi_slices} Axi = {CFG.total_slices} Total')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)

## 2. Data Preparation & Calibrated NLP ReportTeacher

In [ ]:
DATA_DIR = None
for p in [
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
    Path('./data'),
]:
    if p.exists(): DATA_DIR = p; break
if DATA_DIR is None: DATA_DIR = Path('.')
print(f'Data Directory: {DATA_DIR}')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series_df = (
    pd.read_csv(DATA_DIR / 'train_series.csv')
    if (DATA_DIR / 'train_series.csv').exists() else pd.DataFrame()
)
print(f'Total train studies: {len(train_df)}')

PATTERNS = {
    'ACL':              r'\b(acl|anterior cruciate|cruciate ligament)\b',
    'MCL':              r'\b(mcl|medial collateral|collateral ligament)\b',
    'Medial Meniscus':  r'\b(medial meniscus|medial meniscal|posterior horn medial)\b',
    'Lateral Meniscus': r'\b(lateral meniscus|lateral meniscal|posterior horn lateral)\b',
    'Medial OA':        r'\b(medial.{0,20}(osteoarthritis|cartilage loss|narrowing|chondromalacia|osteophyte))\b',
    'Lateral OA':       r'\b(lateral.{0,20}(osteoarthritis|cartilage loss|narrowing|chondromalacia|osteophyte))\b',
    'PF OA':            r'\b(patellofemoral.{0,20}(osteoarthritis|cartilage loss|narrowing|chondromalacia))\b',
    'Effusion':         r'\b(effusion|joint fluid|distension)\b',
    'Synovitis':        r'\b(synovitis|synovial thickening|synovial proliferation)\b',
    "Baker's":          r'\b(baker|popliteal cyst|popliteal bursa)\b',
    'Contusion':        r'\b(contusion|bone bruise|bone marrow edema|marrow edema|subchondral edema)\b',
    'Fracture':         r'\b(fracture|microfracture|cortical disruption|impaction fracture)\b',
}
INJURY   = r'(tear|sprain|rupture|defect|abnormality|edema|loss|thickening|cyst|fracture|bruise|narrowing|edematous)'
NEGATION = r'\b(no|without|intact|unremarkable|negative|free of|absence of|normal|uninjured)\b'

def regex_targets(report_text):
    out = {c: 0.0 for c in CFG.target_cols}
    if not isinstance(report_text, str): return out
    text = report_text.lower()
    for sent in re.split(r'[.\n;]', text):
        for tgt, pat in PATTERNS.items():
            if out[tgt]: continue
            if re.search(pat, sent):
                if re.search(INJURY, sent) or tgt in ['Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']:
                    if not re.search(NEGATION, sent):
                        out[tgt] = 1.0
    return out

class ReportTeacher:
    def __init__(self):
        self.vec = TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english')
        self.models = {}
        self.priors = {}

    def fit(self, df):
        reports = df['Report'].fillna('').astype(str).tolist()
        X = self.vec.fit_transform(reports)
        rule_df = pd.DataFrame([regex_targets(r) for r in reports])
        for c in CFG.target_cols:
            if c in df.columns and df[c].notna().any():
                mask = df[c].notna()
                y = df.loc[mask, c].values.astype(int)
                self.priors[c] = float(np.clip(y.mean(), 0.01, 0.99))
                if len(np.unique(y)) > 1:
                    m = CalibratedClassifierCV(LogisticRegression(C=0.5, max_iter=200), cv=3)
                    m.fit(X[mask.values], y)
                    self.models[c] = m
            else:
                y = rule_df[c].values.astype(int)
                self.priors[c] = float(np.clip(y.mean(), 0.01, 0.99))

    def predict_proba(self, df):
        reports = df['Report'].fillna('').astype(str).tolist()
        X = self.vec.transform(reports)
        probs = np.zeros((len(df), len(CFG.target_cols)), dtype=np.float32)
        for i, c in enumerate(CFG.target_cols):
            if c in self.models:
                probs[:, i] = self.models[c].predict_proba(X)[:, 1]
            else:
                probs[:, i] = self.priors.get(c, 0.1)
        return probs

teacher = ReportTeacher()
teacher.fit(train_df)
teacher_probs = teacher.predict_proba(train_df)
teacher_cols = [f'teacher_{c}' for c in CFG.target_cols]
for i, col in enumerate(teacher_cols):
    train_df[col] = teacher_probs[:, i]

for c in CFG.target_cols:
    train_df[c] = train_df[c].fillna(0.0).astype(np.float32)

strat_proxy = (train_df[CFG.target_cols].sum(axis=1) > 0).astype(int).values
print(f'ReportTeacher ready. Positive proxy studies: {strat_proxy.sum()}/{len(train_df)}')

## 3. Asymmetric 3-View DICOM Dataset with CLAHE (320x320)

In [ ]:
PLANE_KEYWORDS = {
    'sagittal': ['sag'],
    'coronal':  ['cor'],
    'axial':    ['axi', 'tra'],
}

def apply_clahe(arr_uint8):
    if not CFG.use_clahe:
        return arr_uint8
    if HAS_CV2:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(arr_uint8)
    else:
        img = Image.fromarray(arr_uint8, mode='L')
        return np.array(ImageOps.equalize(img))

def read_dcm(path):
    try:
        dcm = pydicom.dcmread(str(path))
        arr = dcm.pixel_array.astype(np.float32)
        arr = arr * float(getattr(dcm, 'RescaleSlope', 1)) + float(getattr(dcm, 'RescaleIntercept', 0))
        lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
        arr = np.clip(arr, lo, hi)
        if hi > lo: arr = (arr - lo) / (hi - lo) * 255.0
        arr = arr.astype(np.uint8)
        arr = apply_clahe(arr)
        return arr
    except Exception:
        return np.zeros(CFG.image_size, dtype=np.uint8)

def sample_slices(files, n):
    if not files:
        return [np.zeros(CFG.image_size, dtype=np.uint8)] * n
    idx = (np.linspace(0, len(files)-1, n, dtype=int)
           if len(files) >= n
           else list(range(len(files))) + [len(files)-1] * (n - len(files)))
    return [read_dcm(files[i]) for i in idx]

class KneeAdaptive3ViewDataset(Dataset):
    def __init__(self, df, series_df, images_dir, transform, is_train=True):
        self.df        = df.reset_index(drop=True)
        self.series_df = series_df
        self.img_dir   = Path(images_dir)
        self.tf        = transform
        self.is_train  = is_train

    def __len__(self): return len(self.df)

    def _plane_files(self, study_id, keywords):
        files = []
        if not self.series_df.empty:
            for kw in keywords:
                mask = (
                    (self.series_df['StudyInstanceUID'] == study_id) &
                    (self.series_df['Anatomical_Plane'].astype(str).str.lower().str.contains(kw))
                )
                for _, row in self.series_df[mask].iterrows():
                    d = self.img_dir / str(study_id) / str(row['SeriesInstanceUID'])
                    if d.exists(): files.extend(sorted(d.glob('*.dcm')))
        return files

    def _all_files(self, study_id):
        d = self.img_dir / str(study_id)
        return sorted(d.rglob('*.dcm')) if d.exists() else []

    def _to_tensor(self, arr_list):
        return torch.stack([self.tf(Image.fromarray(a).convert('RGB')) for a in arr_list])

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        sid  = row['StudyInstanceUID']
        sag_f = self._plane_files(sid, PLANE_KEYWORDS['sagittal'])
        cor_f = self._plane_files(sid, PLANE_KEYWORDS['coronal'])
        axi_f = self._plane_files(sid, PLANE_KEYWORDS['axial'])
        if sum([bool(sag_f), bool(cor_f), bool(axi_f)]) >= 2:
            all_f  = self._all_files(sid)
            slices = (sample_slices(sag_f or all_f, CFG.sag_slices) +
                      sample_slices(cor_f or all_f, CFG.cor_slices) +
                      sample_slices(axi_f or all_f, CFG.axi_slices))
        else:
            slices = sample_slices(self._all_files(sid), CFG.total_slices)
        images = self._to_tensor(slices)
        item   = {'images': images, 'study_id': str(sid)}
        if self.is_train and all(c in self.df.columns for c in CFG.target_cols):
            gt_arr = row[CFG.target_cols].values.astype(np.float32)
            t_arr  = (row[teacher_cols].values.astype(np.float32)
                      if teacher_cols[0] in self.df.columns else gt_arr)
            item['labels']        = torch.tensor(np.nan_to_num(gt_arr, nan=0.0))
            item['teacher_probs'] = torch.tensor(np.nan_to_num(t_arr,  nan=0.1))
        return item

train_transform = T.Compose([
    T.Resize(CFG.image_size),
    T.RandomHorizontalFlip(0.5),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ColorJitter(brightness=0.08, contrast=0.08),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize(CFG.image_size),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print(f'Dataset configured: {CFG.total_slices} slices @ {CFG.image_size} | CLAHE: {CFG.use_clahe}')

## 4. Model Architecture: EfficientNet-B2 + Attention Pool

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1)

class KneeModel(nn.Module):
    def __init__(self, backbone_name, weights_path=None, is_eval=False):
        super().__init__()
        self.backbone = self._load_backbone(backbone_name, weights_path, is_eval)
        D = self.backbone.num_features   # 1408 for B2
        self.pool = AttentionPool(D)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(D, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, CFG.num_classes),
        )

    @staticmethod
    def _load_backbone(name, wp, is_eval):
        if is_eval:
            return timm.create_model(name, pretrained=False, num_classes=0)
        if wp and Path(wp).exists():
            bb = timm.create_model(name, pretrained=False, num_classes=0)
            sd = torch.load(str(wp), map_location='cpu')
            sd = {k: v for k, v in sd.items() if 'classifier' not in k and 'fc' not in k}
            bb.load_state_dict(sd, strict=False)
            print(f'  Loaded offline weights: {Path(wp).name}')
            return bb
        try:
            bb = timm.create_model(name, pretrained=True, num_classes=0)
            print('  Loaded via timm online')
            return bb
        except Exception:
            bb = timm.create_model(name, pretrained=False, num_classes=0)
            return bb

    def forward(self, images):
        B, S, C, H, W = images.shape
        feats = self.backbone(images.view(B*S, C, H, W))
        feats = feats.view(B, S, -1)
        return self.head(self.pool(feats))

_m = KneeModel(CFG.backbone, WEIGHTS_PATH).to(CFG.device)
print(f'Model initialized: {CFG.backbone} ({sum(p.numel() for p in _m.parameters()):,} parameters)')
del _m; gc.collect(); torch.cuda.empty_cache()

## 5. Masked BCE Loss with Baker's Decoupling & Distillation

In [ ]:
pos_counts = train_df[CFG.target_cols].sum().values
total_samples = len(train_df)
pos_weights = np.clip((total_samples - pos_counts) / np.maximum(pos_counts, 1), 1.0, 10.0)
pos_weights[CFG.bakers_idx] = 0.0  # Decouple Baker's from gradient backprop
POS_WEIGHT_TENSOR = torch.tensor(pos_weights, dtype=torch.float32).to(CFG.device)
print('Loss class weights (Baker=0.0):', [f'{w:.1f}' for w in pos_weights])

def compute_loss(logits, labels, teacher_probs):
    loss_gt = F.binary_cross_entropy_with_logits(
        logits, labels, pos_weight=POS_WEIGHT_TENSOR, reduction='none'
    )
    loss_distill = F.binary_cross_entropy_with_logits(
        logits, teacher_probs, reduction='none'
    )
    loss_distill[:, CFG.bakers_idx] = 0.0
    loss_combined = (1.0 - CFG.distill_weight) * loss_gt + CFG.distill_weight * loss_distill
    return loss_combined.mean()

def train_one_epoch(model, loader, optimizer, scaler, scheduler):
    model.train(); running = 0.0; steps = 0
    optimizer.zero_grad()
    for i, batch in enumerate(tqdm(loader, desc='  train')):
        imgs = batch['images'].to(CFG.device, non_blocking=True)
        lbl  = batch['labels'].to(CFG.device, non_blocking=True)
        t_pr = batch['teacher_probs'].to(CFG.device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            logits = model(imgs)
            loss = compute_loss(logits, lbl, t_pr) / CFG.accum_steps
        if not torch.isfinite(loss): continue
        scaler.scale(loss).backward()
        running += loss.item() * CFG.accum_steps; steps += 1
        if (i + 1) % CFG.accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()
    return running / max(steps, 1)

@torch.no_grad()
def validate(model, loader):
    model.eval(); all_p, all_y = [], []
    for batch in tqdm(loader, desc='  val'):
        imgs = batch['images'].to(CFG.device, non_blocking=True)
        all_p.append(torch.sigmoid(model(imgs)).cpu().numpy())
        all_y.append(batch['labels'].cpu().numpy())
    preds  = np.nan_to_num(np.concatenate(all_p), nan=0.5)
    labels = np.concatenate(all_y)
    # Force constant empirical prior on Baker's
    preds[:, CFG.bakers_idx] = float(np.clip(teacher.priors.get("Baker's", 0.01), 0.005, 0.05))
    
    aucs = {}
    for i, col in enumerate(CFG.target_cols):
        try:
            aucs[col] = roc_auc_score(labels[:, i], preds[:, i]) if len(np.unique(labels[:, i])) > 1 else 0.5
        except Exception:
            aucs[col] = 0.5
    return float(np.mean(list(aucs.values()))), aucs

@torch.no_grad()
def tta_infer(model, loader):
    model.eval(); all_p = []
    for batch in tqdm(loader, desc='  infer'):
        imgs = batch['images'].to(CFG.device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            p0 = torch.sigmoid(model(imgs))
            p1 = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
            p2 = torch.sigmoid(model(torch.rot90(imgs, k=1, dims=[-2,-1])))
            p3 = torch.sigmoid(model(torch.rot90(imgs, k=3, dims=[-2,-1])))
        all_p.append(((p0 + p1 + p2 + p3) / 4.0).cpu().numpy())
    return np.concatenate(all_p)

## 6. 5-Fold Training Loop (320x320 B2)

In [ ]:
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
fold_paths, fold_aucs = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, strat_proxy)):
    print(f'\n{"="*60}\nFOLD {fold+1}/{CFG.n_folds} (320x320 B2 | Seed {CFG.seed})\n{"="*60}')
    df_tr = train_df.iloc[tr_idx]
    df_va = train_df.iloc[va_idx]

    tr_ds = KneeAdaptive3ViewDataset(df_tr, train_series_df, DATA_DIR/'train_series', train_transform, True)
    va_ds = KneeAdaptive3ViewDataset(df_va, train_series_df, DATA_DIR/'train_series', val_transform,   True)
    tr_ld = DataLoader(tr_ds, batch_size=CFG.batch_size, shuffle=True,
                       num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
    va_ld = DataLoader(va_ds, batch_size=CFG.batch_size, shuffle=False,
                       num_workers=CFG.num_workers, pin_memory=True)

    model = KneeModel(CFG.backbone, WEIGHTS_PATH).to(CFG.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    total_steps = CFG.epochs * len(tr_ld) // CFG.accum_steps
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1), eta_min=CFG.lr_min)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.use_amp)

    best_auc, best_ep, best_state = 0.0, 0, None

    for epoch in range(CFG.epochs):
        tr_loss = train_one_epoch(model, tr_ld, optimizer, scaler, scheduler)
        va_auc, per_cls = validate(model, va_ld)
        print(f'  Ep {epoch+1}/{CFG.epochs}  trn_loss={tr_loss:.4f}  val_auc={va_auc:.4f}')
        for col, auc in per_cls.items():
            print(f'    {col:20s} {auc:.4f}')

        if va_auc > best_auc:
            best_auc   = va_auc
            best_ep    = epoch + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    path = f'model_b2_320_seed{CFG.seed}_fold{fold}.pth'
    torch.save(best_state, path)
    fold_paths.append(path); fold_aucs.append(best_auc)
    print(f'  >> Fold {fold+1} Best AUC = {best_auc:.4f} @ Ep{best_ep}')
    del model, optimizer, scheduler, scaler, best_state, tr_ds, va_ds, tr_ld, va_ld
    gc.collect(); torch.cuda.empty_cache()

cv_mean, cv_std = float(np.mean(fold_aucs)), float(np.std(fold_aucs))
print(f'\nV35 (320x320 B2 Seed {CFG.seed}) CV = {cv_mean:.4f} +/- {cv_std:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')

## 7. Test Inference & Submission

In [ ]:
print('Loading test dataset...')
test_csv     = DATA_DIR / 'test.csv'
sample_sub   = DATA_DIR / 'sample_submission.csv'
test_ser_csv = DATA_DIR / 'test_series.csv'

test_df     = pd.read_csv(test_csv if test_csv.exists() else sample_sub)
test_series = pd.read_csv(test_ser_csv) if test_ser_csv.exists() else pd.DataFrame()
print(f'Test studies: {len(test_df)}')

test_ds = KneeAdaptive3ViewDataset(test_df, test_series, DATA_DIR/'test_series',
                                   val_transform, is_train=False)
test_ld = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                     num_workers=0, pin_memory=False)

print(f'Running {len(fold_paths)}-fold 320x320 B2 inference with 4-way TTA...')
b2_320_preds = []
try:
    for fi, mp in enumerate(fold_paths):
        print(f'  320x320 B2 Fold {fi+1} inference...')
        m = KneeModel(CFG.backbone, weights_path=None, is_eval=True).to(CFG.device)
        m.load_state_dict(torch.load(mp, map_location=CFG.device))
        b2_320_preds.append(tta_infer(m, test_ld))
        del m; gc.collect(); torch.cuda.empty_cache()
    final = np.mean(b2_320_preds, axis=0)
except Exception as e:
    print(f'Inference fallback: {e}')
    final = np.full((len(test_df), CFG.num_classes), 0.5)

final = np.clip(np.nan_to_num(final, nan=0.5), 0.001, 0.999)
final[:, CFG.bakers_idx] = float(np.clip(teacher.priors.get("Baker's", 0.01), 0.005, 0.05))

sub = pd.DataFrame(final, columns=CFG.target_cols)
sub.insert(0, 'StudyInstanceUID', test_df['StudyInstanceUID'].values)
sub.to_csv('submission.csv', index=False)
sub.to_csv('submission_b2_320.csv', index=False)

print(f'Submission shape: {sub.shape}')
print(f'Any NaN: {sub.isnull().any().any()}')
assert os.path.exists('submission.csv')
print('Done! V35 (320x320 EfficientNet-B2 Foundation) completed successfully.')